In [25]:
import pandas as pd
import numpy as np
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

In [26]:
distance_matrix = np.load('../data/distance_matrix.npy')
sample = pd.read_csv('../data/sample_stops.csv')

In [27]:
distance_matrix = distance_matrix.astype(int)
manager = pywrapcp.RoutingIndexManager(
    len(distance_matrix), #number of locations
    3,                    #number of vehicles
    0                     #depot index
)
routing = pywrapcp.RoutingModel(manager)

In [28]:
def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return distance_matrix[from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

In [29]:
print(len(sample))

21


In [30]:
sample['demand'] = np.random.randint(1,6, size=len(sample))
sample.loc[0, 'demand'] = 0

In [31]:
def demand_callback(from_index):
    from_node = manager.IndexToNode(from_index)
    return int(sample.iloc[from_node]['demand'])
demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)

In [32]:
print(sample['demand'].sum())

60


In [33]:
routing.AddDimensionWithVehicleCapacity(
    demand_callback_index, 
    0,
    [25, 25, 25],
    True,
    'Capacity'
)

True

In [34]:
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_parameters.time_limit.seconds = 30

In [35]:
solution = routing.SolveWithParameters(search_parameters)

In [36]:
if solution:
    print(solution)
else:
    print("No solution found")

Assignment(Capacity0 (0) | Capacity1 (6) | Capacity2 (5) | Capacity3 (4) | Capacity4 (0) | Capacity5 (23) | Capacity6 (22) | Capacity7 (20) | Capacity8 (16) | Capacity9 (24) | Capacity10 (13) | Capacity11 (8) | Capacity12 (3) | Capacity13 (0) | Capacity14 (21) | Capacity15 (19) | Capacity16 (15) | Capacity17 (11) | Capacity18 (6) | Capacity19 (4) | Capacity20 (0) | Capacity21 (0) | Capacity22 (0) | Capacity23 (10) | Capacity24 (25) | Capacity25 (25) | Nexts0 (4) | Nexts1 (23) | Nexts2 (1) | Nexts3 (2) | Nexts4 (3) | Nexts5 (24) | Nexts6 (5) | Nexts7 (6) | Nexts8 (7) | Nexts9 (25) | Nexts10 (8) | Nexts11 (10) | Nexts12 (11) | Nexts13 (12) | Nexts14 (9) | Nexts15 (14) | Nexts16 (15) | Nexts17 (16) | Nexts18 (17) | Nexts19 (18) | Nexts20 (19) | Nexts21 (13) | Nexts22 (20) | Active0 (1) | Active1 (1) | Active2 (1) | Active3 (1) | Active4 (1) | Active5 (1) | Active6 (1) | Active7 (1) | Active8 (1) | Active9 (1) | Active10 (1) | Active11 (1) | Active12 (1) | Active13 (1) | Active14 (1) | Act

In [41]:
for vehicle_id in range(3):
    index = routing.Start(vehicle_id)
    route = []
    route_distance = 0

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route.append(sample.iloc[node]['Address'])
        previous_index = index
        index = solution.Value(routing.NextVar(index))
        route_distance += distance_matrix[manager.IndexToNode(previous_index)][manager.IndexToNode(index)]
        
    route.append(sample.iloc[manager.IndexToNode(index)]['Address'])
    print(f"Driver {vehicle_id + 1}:")
    for stop in route:
        print(f" -> {stop}")
    print(f" Total Distance: {round(route_distance/1000, 2)} km")
    print()

Driver 1:
 -> 4520 Bullock Farm Rd
 -> 3100 Highwoods Blvd STE 115
 -> 9400 Brier Creek Pkwy STE 203
 -> 7201 Creedmoor Rd STE 150
 -> 6675 Falls Of Neuse Rd STE 115
 -> 4520 Bullock Farm Rd
 Total Distance: 63.63 km

Driver 2:
 -> 4520 Bullock Farm Rd
 -> 1010 Main Campus Dr STE 150
 -> 547 Pylon Dr
 -> 6411 Lakecrest Dr
 -> 226 E Martin St
 -> 6308 Angus Dr STE E
 -> 6837 Falls Of Neuse Rd STE 206
 -> 6303 Chapel Hill Rd
 -> 3214 Student Ln
 -> 4520 Bullock Farm Rd
 Total Distance: 118.92 km

Driver 3:
 -> 4520 Bullock Farm Rd
 -> 5811 Poyner Village Pkwy
 -> 4200 Hillsborough St
 -> 3225 Capital Blvd
 -> 4325 Glenwood Ave UNIT B
 -> 4131 Parklake Ave STE 200
 -> 701 Corporate Center Dr STE 185
 -> 3757 Benson Dr
 -> 111 E North St
 -> 4520 Bullock Farm Rd
 Total Distance: 92.44 km



In [ ]:
results = []
for vehicle_id in range(3):
    index = routing.Start(vehicle_id)
    stop_number = 0
    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        results.append({
        'vehicle': vehicle_id + 1,
        'stop_sequence': stop_number,
        'node': node, 
        'address': sample.iloc[node]['Address'],
        'latitude': sample.iloc[node]['latitude'],
        'longitude': sample.iloc[node]['long